In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [3]:
from solarrpy import seasonalClearsky
from solarrpy import seasonalModel

In [4]:
"""
import cdsapi

dataset = "cams-solar-radiation-timeseries"
request = {
    "sky_type": "observed_cloud",
    "location": {"longitude": 11.3426, "latitude": 44.4949},
    "altitude": ["71"],
    "date": ["2005-01-01/2026-03-31"],
    "time_step": "1day",
    "time_reference": "true_solar_time",
    "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()
"""

'\nimport cdsapi\n\ndataset = "cams-solar-radiation-timeseries"\nrequest = {\n    "sky_type": "observed_cloud",\n    "location": {"longitude": 11.3426, "latitude": 44.4949},\n    "altitude": ["71"],\n    "date": ["2005-01-01/2026-03-31"],\n    "time_step": "1day",\n    "time_reference": "true_solar_time",\n    "data_format": "csv"\n}\n\nclient = cdsapi.Client()\nclient.retrieve(dataset, request).download()\n'

In [ ]:
df = pd.read_csv("../data/CAMS_data/CAMS_data_Bologna.csv")
#df = pd.read_csv("../data/Bologna.csv")
spec = {
    'target': 'GHI',
    'coords': {
        "lat": 44.4949,
        "lon": 11.3426,
        "alt": 71
    },
    'data': df
}

In [ ]:
# ----------------------------------------------------------------------------
# NOTE: This script assumes the existence of the previously translated classes/functions:
# SeasonalClearsky, control_seasonalClearsky, clearsky_outliers
# 
# It also assumes a pre-existing `spec` object/dictionary that contains the dataset.
# For example:
# spec = {
#     'data': pd.DataFrame({'GHI': [...], 'date': [...], 'clearsky': [...], 'n': [...]}),
#     'coords': {'lat': 45.0},
#     'target': 'GHI'
# }
# ----------------------------------------------------------------------------

# ==========================================
# Inputs
# ==========================================

# Control parameters
control = seasonalClearsky.control_seasonalClearsky(
    orders=1, order_H0=1, periods=365, 
    include_intercept=True, include_trend=False,
    delta0=1.4, lower=0, upper=3, by=0.001, ntol=0, quiet=False
)

# Extracting from `spec` object
data_all = spec['data'].copy()
data_all['date'] = pd.to_datetime(data_all['date'])
lat = spec['coords']['lat']
alt = spec['coords']['alt']
target_col = spec['target']

# Command parameters
plot_data = False
test_outputs = False

computed_values = []
Ct_arrays = {}

for year_i in range(2013, 2023):
    mask = data_all['date'].dt.year <= year_i
    data = data_all.loc[mask].copy()
    
    model_coefficients = {}

    print(f"\nRunning model with data up to {year_i}-12-31 ({len(data)} rows)")

    GHI = data['GHI']
    date = data['date']
    clearsky = data['clearsky']
    H0 = data['H0']

    # ==========================================
    # Fit the clear sky model
    # ==========================================

    # Initialize the model 
    clearsky_model = seasonalClearsky.SeasonalClearsky(control=control)

    # Fit the parameters
    clearsky_model.fit(GHI, date, lat, clearsky, alt=alt)
    #print(clearsky_model)
    
    # Predictions
    data['Ct'] = clearsky_model.predict(n=data['n'], newdata=data)
    #print(f"Ct: [{min(data['Ct']):.2f},{max(data['Ct']):.2f}], GHI: [{min(data['GHI']):.2f},{max(data['GHI']):.2f}]")
    #df = pd.DataFrame(data[['n', 'GHI', 'Ct']])
    #display(df.iloc[np.where(df['GHI'] >= df['Ct'])])

    # Save the model coefficients
    delta0, delta1, delta2, delta3 = clearsky_model._model.params.values[:4]
    delta0_err, delta1_err, delta2_err, delta3_err = clearsky_model._model.bse.values[:4]

    computed_values.append({
        'Year': year_i,
        'delta0': delta0, 'delta1': delta1, 'delta2': delta2, 'delta3': delta3,
        'delta0_err': delta0_err, 'delta1_err': delta1_err, 
        'delta2_err': delta2_err, 'delta3_err': delta3_err,
    })

    Ct_arrays[f'{year_i}'] = data['Ct'].values

    # ==========================================
    # Plotting
    # ==========================================
    if plot_data:
        # Filter data between dates
        df_plot = data

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Plot 1: CAMS vs GHI
        axes[0].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[0].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[0].set_xlabel('Day of the year')
        axes[0].set_ylabel('Clear sky')
        axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[0].grid(True, linestyle='--', alpha=0.7)

        # Plot 2: Fitted vs CAMS
        axes[1].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[1].plot(df_plot['n'], df_plot['clearsky'], linestyle=' ', marker='.', color='blue', label='CAMS')
        axes[1].set_xlabel('Day of the year')
        axes[1].set_ylabel('Clear sky')
        axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[1].grid(True, linestyle='--', alpha=0.7)

        # Plot 3: Fitted vs GHI
        axes[2].plot(df_plot['n'], df_plot['Ct'], linestyle=' ', marker='.', color='red', label='Fitted')
        axes[2].plot(df_plot['n'], df_plot['GHI'], linestyle=' ', marker='.', color='black', label='GHI')
        axes[2].set_xlabel('Day of the year')
        axes[2].set_ylabel('Clear sky')
        axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
        axes[2].grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.show()

    if test_outputs:
        # ==========================================
        # Test: imputed outliers
        # ==========================================
        # Impute outliers
        outliers = seasonalClearsky.clearsky_outliers(data[target_col], data['Ct'], data['date'], quiet=True)
        data[target_col] = outliers['x']
        
        # Test tolerance parameter
        print("\033[1;35m---------------\033[0m \033[1;32m  Test clearskyModel_control and clearskyModel_fit \033[1;35m---------------\033[0m")
        
        passed_ntol = outliers['n'] <= control['ntol']
        msg_ntol = "\033[1;32mPassed\033[0m!\n" if passed_ntol else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the number of outliers imputed is below {control['ntol']}...({outliers['n']}) {msg_ntol}")
    
        # ==========================================
        # Test: delta parameter
        # ==========================================
        
        # Test delta parameter (Accessing mangled private attribute)
        delta = clearsky_model.delta    
        test_delta = (delta > control['lower']) and (delta < control['upper'])
        msg_delta = "\033[1;32mPassed\033[0m!\n" if test_delta else "\033[1;31mNOT passed\033[0m! \n"
        
        print(f"Check if the parameter delta is inside lower ({control['lower']}) and upper ({control['upper']})...({delta}) {msg_delta}")
        
        # ==========================================
        # Test: order of seasonal components
        # ==========================================
        
        # Count the number of parameters 
        n_params_target = 1 if control['include_intercept'] else 0
        n_params_target += 1 if control['include_trend'] else 0
        n_params_target += control['orders'] * 2
        n_params_target += control['order_H0']
        
        # Test if the number of parameters is correct 
        n_params = len(clearsky_model._model.params)
        
        # Print result 
        msg_params = "\033[1;32mPassed\033[0m!\n" if n_params == n_params_target else "\033[1;31mNOT passed\033[0m! \n"
        print(f"Check if the number of parameters is equal to {n_params_target}...({n_params}) {msg_params}")
        
        # ==========================================
        # Differential
        # ==========================================
        
        # Note differential when a trend is true is not implemented
        dt = 0.05
        n0 = 34

        num_diff = (clearsky_model.predict(n=n0 + dt) - clearsky_model.predict(n=n0)) / dt
        ana_diff = clearsky_model.differential(n=n0)
        
        print(f"Numerical Differential: \n{num_diff}")
        print(f"Analytical Differential: \n{ana_diff}")


Running model with data up to 2013-12-31 (3287 rows)

Running model with data up to 2014-12-31 (3652 rows)

Running model with data up to 2015-12-31 (4017 rows)

Running model with data up to 2016-12-31 (4383 rows)

Running model with data up to 2017-12-31 (4748 rows)

Running model with data up to 2018-12-31 (5113 rows)

Running model with data up to 2019-12-31 (5478 rows)

Running model with data up to 2020-12-31 (5844 rows)

Running model with data up to 2021-12-31 (6209 rows)

Running model with data up to 2022-12-31 (6574 rows)


In [7]:
# Save scalars as a Parquet file for exact type preservation
computed_values_df = pd.DataFrame(computed_values)
computed_values_df.to_parquet('../results/A1_deltas.parquet', engine='pyarrow')

# Save arrays using NumPy's compressed format
np.savez_compressed('../results/A1_Ct_arrays.npz', **Ct_arrays)

In [8]:
display(computed_values_df)

,Year,delta0,delta1,delta2,delta3,delta0_err,delta1_err,delta2_err,delta3_err
0,2013,-1.171128,0.945874,0.011138,0.509010,0.316482,0.042455,0.031483,0.181048
1,2014,-1.081158,0.935855,0.025053,0.450270,0.299062,0.040118,0.029750,0.171081
2,2015,-1.311852,0.967189,0.000488,0.585216,0.282857,0.037944,0.028138,0.161811
3,2016,-1.237380,0.956716,0.006169,0.540969,0.272675,0.036579,0.027126,0.155988
4,2017,-1.155944,0.945600,0.019045,0.496049,0.261399,0.035066,0.026004,0.149537
5,2018,-1.126893,0.940606,0.022330,0.483482,0.252972,0.033935,0.025165,0.144715
6,2019,-1.297742,0.963243,0.012548,0.587640,0.244725,0.032829,0.024345,0.139997
7,2020,-1.321076,0.967355,0.011603,0.601091,0.236216,0.031688,0.023499,0.135131
8,2021,-1.494182,0.991635,-0.000809,0.700337,0.229815,0.030829,0.022862,0.131469
9,2022,-1.662377,1.014553,-0.016788,0.795776,0.224359,0.030097,0.022319,0.128348
